# Evaluation of the intersections of the UKB analyses
### 1. Import of ukb_bulk_scan (created with scan_ukb.py)

In [ ]:
from os.path import split

import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

In [ ]:
csv_name = "out/csv/20260121_ukb_bulk_with_Proteomics.csv"
df = pd.read_csv(csv_name)

n_participants = df.shape[0]
print(f"Number of participants: {n_participants}")

In [ ]:
    categories = [
    col for col in df.columns
    if col not in {"id", "anon_id"}
]

print(f"Number of categories: {len(categories)}. Categories: {categories}")

In [ ]:
# Playing with data
print(df["Carotid Ultrasound/Carotid artery (left)"].sum())
print(df["Carotid Ultrasound/Carotid artery (right)"].sum())

true_counts = df[categories].sum()

top_category = true_counts.idxmax()
top_count = true_counts.max()

print(f"\nCategory with most TRUEs: {top_category}")
print(f"Number of participants: {top_count}")
print(f"Percentage: {top_count / n_participants:.2%}")

print(f"\nTop 10 categories:\n{true_counts.sort_values(ascending=False).head(10)}")


### 2. Intersection Calcs

#### 2.1 - 2 categories

In [ ]:
cat1 = "Carotid Ultrasound/Carotid artery (left)"
cat2 = "Proteomics"

n_cat1 = df[cat1].sum()
n_cat2 = df[cat2].sum()

n_intersect = (df[cat1] & df[cat2]).sum()
print(n_intersect)

#### 2.2 - 3+ categories

In [ ]:
cats = ["Carotididc/Pics", "Imaging/MRI", "Proteomics/Olink"]

intersection_count = df[cats].all(axis=1).sum()
intersection_count

### 3 Plotting

#### 3.1 - 2 categories

In [ ]:
plt.style.use("default")
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "text.color": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
})

In [ ]:
def short_name(cat: str) -> str:
    return cat.split("/")[-1].replace(" ", "_")

labels = [short_name(cat1), short_name(cat2), f"intersect"]
values = [n_cat1, n_cat2, n_intersect]

bars = plt.bar(labels, values)

plt.bar_label(
    bars,
    labels=[f"{v:,}" for v in values],
    padding=3,
    fontsize=11,
    fontweight = "heavy",
    color="blue"
)

plt.axhline(
    n_participants,
    linestyle="--",
    color="red",
)

plt.text(
    x=len(labels) - 0.3,          # rechts am Plot
    y=42000,
    s=f"Total participants (N={n_participants:,})",
    rotation=90,
    va="bottom",
    ha="right",
    color="red",
    fontsize=11
)

plt.ylabel("Number of participants")
plt.title(f"{short_name(cat1)} ∩ {short_name(cat2)}")
# plt.legend()
plt.tight_layout()

date_str = Path(csv_name).stem[:8]

out_name = f"{date_str}_{short_name(cat1)}_{short_name(cat2)}_intersect.png"
out_dir = Path("out/plots")
plt.savefig(out_dir / out_name, dpi=600)
plt.show()

print(f"Saved figure as {out_name}.")

#### 4: Filtering image_counts for main axis images

In [ ]:
import pandas as pd

df = pd.read_csv("/Users/Maxi/Desktop/PlaqueRisk_data/main_axis_count.csv")

# 1) Wie oft beide FALSE sind
both_false = ((df["Img left"] == False) & (df["Img right"] == False)).sum()

# 2) In wie vielen Spalten (pro Zeile) mindestens einmal > 1 steht
at_least_one_gt1 = (
    (pd.to_numeric(df["Img left"], errors="coerce").fillna(0) > 1) |
    (pd.to_numeric(df["Img right"], errors="coerce").fillna(0) > 1)
).sum()

at_least_two_gt1 = (
    (pd.to_numeric(df["Img left"], errors="coerce").fillna(0) > 2) |
    (pd.to_numeric(df["Img right"], errors="coerce").fillna(0) > 2)
).sum()

print(f"Beide FALSE: {both_false}")
print(f"Mind. eine Seite > 1: {at_least_one_gt1}")
print(f"Mind. eine Seite > 2: {at_least_two_gt1}")

In [ ]:
from pathlib import Path

BASE = Path("/Users/Maxi/Desktop/PlaqueRisk_data")
LEFT_FILE  = BASE / "left_pxlsum.txt"
RIGHT_FILE = BASE / "right_pxlsum.txt"
OUT_FILE   = BASE / "main_axis_images.txt"

def extract_dcm_name(line: str):
    # split by the separators used in your files
    parts = [p.strip() for p in line.split("::")]
    if len(parts) < 2:
        return None
    dcm = parts[1]
    return dcm if dcm.endswith(".dcm") else None

images = set()

for p in (LEFT_FILE, RIGHT_FILE):
    with p.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            dcm = extract_dcm_name(line)
            if dcm:
                images.add(dcm)

# write unique, sorted
with OUT_FILE.open("w", encoding="utf-8") as f:
    for name in sorted(images):
        f.write(name + "\n")

print(f"Found {len(images)} unique image names.")
print(f"Saved to: {OUT_FILE}")

In [ ]:
from pathlib import Path
import pandas as pd

BASE = Path("/Users/Maxi/Desktop/PlaqueRisk_data")

IMAGES_FILE = BASE / "main_axis_images.txt"
CSV_IN      = BASE / "image_level.csv"
CSV_OUT     = BASE / "corr_image_level.csv"

with IMAGES_FILE.open("r") as f:
    main_images = set(line.strip() for line in f if line.strip())

df = pd.read_csv(CSV_IN)
df["image_name"] = df["image_path_in_zip"].apply(lambda p: Path(p).name)
df_filt = df[df["image_name"].isin(main_images)].copy()
df_filt = df_filt.drop(columns=["image_name"])
df_filt.to_csv(CSV_OUT, index=False)

print(f"Gefilterte Zeilen: {len(df_filt)}")
print(f"Saved to: {CSV_OUT}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(CSV_OUT)

pid_plaque = (
    df.groupby("participant_id")["n_boxes"]
      .apply(lambda x: (x > 0).any())
      .reset_index(name="plaque_detected")
)

counts = pid_plaque["plaque_detected"].value_counts()
n_plaque = counts.get(True, 0)
n_no_plaque = counts.get(False, 0)

print(f"Plaque detected (≥1 Bild mit n_boxes>0): {n_plaque}")
print(f"No plaque detected: {n_no_plaque}")

labels = ["Plaque detected", "No plaque detected"]
values = [n_plaque, n_no_plaque]

plt.figure()
plt.bar(labels, values)
plt.ylabel("Number of PIDs")
plt.title("Plaque detection per PID (main axis images)")
plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path("/Users/Maxi/Desktop/PlaqueRisk_data")
COUNT_CSV = BASE / "main_axis_count.csv"
IMAGE_CSV = BASE / "corr_image_level.csv"

df_count = pd.read_csv(COUNT_CSV)

def to_count(series: pd.Series) -> pd.Series:
    # "False", "", NaN -> 0 ; numbers/strings -> int
    return pd.to_numeric(series, errors="coerce").fillna(0).astype(int)

left_n = to_count(df_count["Img left"])
right_n = to_count(df_count["Img right"])

valid_pids = df_count.loc[(left_n > 0) & (right_n > 0), "PID"].astype(str)
valid_pid_set = set(valid_pids)

print(f"Total PIDs in count file: {df_count.shape[0]}")
print(f"Valid PIDs (left>0 & right>0): {len(valid_pid_set)}")
print(f"Excluded (missing left OR right): {int(((left_n==0) | (right_n==0)).sum())}")

# ----------------------------
# 2) Load image-level CSV and filter to valid PIDs
# ----------------------------
df_img = pd.read_csv(IMAGE_CSV)

# ensure comparable dtype
df_img["participant_id"] = df_img["participant_id"].astype(str)

df_img_filt = df_img[df_img["participant_id"].isin(valid_pid_set)].copy()

print(f"Rows in image file: {df_img.shape[0]}")
print(f"Rows after PID filter: {df_img_filt.shape[0]}")
print(f"Unique PIDs after filter: {df_img_filt['participant_id'].nunique()}")

# ----------------------------
# 3) Plaque detected per PID: any n_boxes > 0
# ----------------------------
df_img_filt["n_boxes"] = pd.to_numeric(df_img_filt["n_boxes"], errors="coerce").fillna(0).astype(int)

pid_plaque = (
    df_img_filt.groupby("participant_id")["n_boxes"]
               .apply(lambda x: (x > 0).any())
               .reset_index(name="plaque_detected")
)

counts = pid_plaque["plaque_detected"].value_counts()
n_plaque = int(counts.get(True, 0))
n_no_plaque = int(counts.get(False, 0))

print(f"Plaque detected (per PID, any n_boxes>0): {n_plaque}")
print(f"No plaque detected (per PID): {n_no_plaque}")

# ----------------------------
# 4) Plot
# ----------------------------
plt.figure()
plt.bar(["Plaque detected", "No plaque detected"], [n_plaque, n_no_plaque])
plt.ylabel("Number of PIDs")
plt.title("Plaque detection per PID\n(only PIDs with left & right images present)")
plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import pandas as pd

BASE = Path("/Users/Maxi/Desktop/PlaqueRisk_data")
COUNT_CSV = BASE / "main_axis_count.csv"

df = pd.read_csv(COUNT_CSV)

# Zählt Zeilen, in denen in irgendeiner Spalte ein "False" vorkommt
false_rows = (
    df.astype(str)
      .apply(lambda row: row.str.strip().str.lower().eq("false").any(), axis=1)
      .sum()
)

print(f"Zeilen mit mindestens einem 'False': {false_rows}")

In [ ]:
from pathlib import Path
import pandas as pd

BASE = Path("/Users/Maxi/Desktop/PlaqueRisk_data")
IMAGE_CSV = BASE / "corr_image_level.csv"
OUT_CSV   = BASE / "pid_plaque_status.csv"

# --------------------------------
# 1) Load image-level data
# --------------------------------
df = pd.read_csv(IMAGE_CSV)

df["participant_id"] = df["participant_id"].astype(str)
df["n_boxes"] = pd.to_numeric(df["n_boxes"], errors="coerce").fillna(0).astype(int)

# --------------------------------
# 2) Plaque status per PID
#    True if ANY image has n_boxes > 0
# --------------------------------
pid_plaque = (
    df.groupby("participant_id")["n_boxes"]
      .apply(lambda x: (x > 0).any())
      .reset_index()
      .rename(columns={
          "participant_id": "PID",
          "n_boxes": "Plaque"
      })
)

pid_plaque["Plaque"] = pid_plaque["Plaque"].astype(bool)

# --------------------------------
# 3) Save
# --------------------------------
pid_plaque.to_csv(OUT_CSV, index=False)

print(f"Saved: {OUT_CSV}")
print(pid_plaque['Plaque'].value_counts())

In [ ]:
print(16-9)